# NB_LIBRO_08 · Figuras y experimentos del capítulo 6

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 6, «Aprendizaje profundo para visión»

**Qué hace.** Con los modelos ya entrenados de los dos proyectos y con sus datos:

1. aplica tres filtros de convolución a un racimo;
2. muestra los mapas de activación del clasificador YOLO11m en cinco profundidades;
3. muestra ejemplos de la salida del clasificador y del detector;
4. calcula la curva de precisión y sensibilidad del detector YOLO11s y su precisión media con diez umbrales de IoU, en toda la validación y en las imágenes sin gemela en entrenamiento;
5. mide el acuerdo, en IoU, entre las dos anotaciones de las imágenes repetidas del conjunto PIO;
6. entrena YOLO11m desde cero y lo compara con el preentrenado (transferencia de aprendizaje);
7. cuenta parámetros y capas de los modelos YOLO11 y YOLO26.

**Usa (solo lectura):** `PALMA_AFRICANA_YOLO` (datos procesados y `05_MODELOS/yolo11m_cls_seed2026/best.pt`), `AgroUTMACH_PIX` (validación de PIO y `03_MODELOS/AgroUTMACH_PIX_YOLO11s_PIO_best.pt`) y `AGROVISION_LIBRO/03_VALORES/pio_vecino_val_train.csv`. No modifica nada de los proyectos.

**Cómo ejecutarlo:** GPU (L4 recomendada) y *Ejecutar todo*. Tarda unos 35 a 45 minutos; la mayor parte son siete entrenamientos. Si una sección falla, las demás continúan y el error queda anotado en el archivo de valores. Si Colab se desconecta, vuelva a ejecutar todo: los entrenamientos terminados se reutilizan. Suba al chat el zip que se descarga al final.

In [ ]:
%pip install -q ultralytics==8.4.172 scikit-learn scipy

In [ ]:
import os, re, json, math, zipfile, time, shutil, gc, random, traceback
from pathlib import Path
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
LOCAL = Path(os.environ.get('AGROVISION_LOCAL', '/content'))
PALMA = BASE / 'PALMA_AFRICANA_YOLO'
DATOS = PALMA / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
MOD_PALMA = PALMA / '05_MODELOS' / 'yolo11m_cls_seed2026' / 'best.pt'
PIX = BASE / 'AgroUTMACH_PIX'
PIO = PIX / '01_DATASET_PIO' / 'PIO_TRAIN_VAL' / 'data'
MOD_PIO = PIX / '03_MODELOS' / 'AgroUTMACH_PIX_YOLO11s_PIO_best.pt'
DIR_FIG = BASE / 'AGROVISION_LIBRO' / '02_FIGURAS' / 'cap06'
DIR_VAL = BASE / 'AGROVISION_LIBRO' / '03_VALORES'
DIR_EXP = DIR_VAL / 'cap06_transferencia'
for d in (DIR_FIG, DIR_VAL, DIR_EXP):
    d.mkdir(parents=True, exist_ok=True)
for p in (DATOS, MOD_PALMA, PIO / 'images' / 'val', MOD_PIO):
    print('OK   ' if p.exists() else 'FALTA', p)

import torch, ultralytics
from ultralytics import YOLO
from sklearn.metrics import accuracy_score, f1_score
HAY_GPU = torch.cuda.is_available()
DISP = 0 if HAY_GPU else 'cpu'
GPU = torch.cuda.get_device_name(0) if HAY_GPU else 'sin GPU'
print('GPU:', GPU, '| PyTorch', torch.__version__, '| Ultralytics', ultralytics.__version__)
assert HAY_GPU or not EN_COLAB, 'Active una GPU en Entorno de ejecución > Cambiar tipo de entorno.'

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)


ERRORES = {}
CLASES_ORD = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']


def seccion(nombre, fn):
    # Ejecuta una sección y, si falla, anota el error y deja que las demás continúen.
    t0 = time.time()
    try:
        fn()
        print(f'[{nombre}] terminada en {time.time() - t0:.0f} s')
    except Exception:
        ERRORES[nombre] = traceback.format_exc()[-1500:]
        print(f'[{nombre}] ERROR\n{ERRORES[nombre]}')
    plt.close('all')

## 1. La convolución sobre un racimo

In [ ]:
def s_convolucion():
    ruta = sorted((DATOS / 'test' / 'ripe').glob('*.jpg'))[0]
    rgb = recorte_cuadrado(leer_rgb(ruta, 768), 384)
    g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255
    filtros = [('Bordes verticales', [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
               ('Bordes horizontales', [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
               ('Puntos y manchas', [[0, -1, 0], [-1, 4, -1], [0, -1, 0]])]
    fig, ejes = plt.subplots(1, 4, figsize=(ANCHO_IN, 2.2), gridspec_kw={'wspace': 0.06})
    ejes[0].imshow(g, cmap='gray', vmin=0, vmax=1); ejes[0].set_title('a) Imagen en grises', loc='left', fontweight='bold', fontsize=7)
    for ax, letra, (nombre, k) in zip(ejes[1:], 'bcd', filtros):
        r = np.abs(cv2.filter2D(g, cv2.CV_32F, np.array(k, np.float32)))
        ax.imshow(r, cmap='gray', vmin=0, vmax=float(np.percentile(r, 99)))
        ax.set_title(f'{letra}) {nombre}', loc='left', fontweight='bold', fontsize=7)
        ax.set_xlabel('\n'.join('  '.join(f'{v:>2d}' for v in fila) for fila in k), fontsize=6.5, family='DejaVu Sans Mono', color=TINTA_2)
    ejes[0].set_xlabel(f'{g.shape[1]} × {g.shape[0]} píxeles', fontsize=6.5, color=TINTA_2)
    for ax in ejes:
        sin_ejes(ax)
    guardar(fig, DIR_FIG / 'Fig_6_2_convolucion.png'); plt.show()
    VALORES['convolucion'] = {'imagen': ruta.name}

seccion('convolucion', s_convolucion)

## 2. Qué calcula cada capa del clasificador

In [ ]:
def s_mapas():
    ruta = sorted((DATOS / 'test' / 'ripe').glob('*.jpg'))[0]
    rgb = recorte_cuadrado(leer_rgb(ruta, 768), 384)
    m = YOLO(str(MOD_PALMA))
    red = m.model.float().eval()
    capas = list(red.model)
    indices = sorted(set([0, 2, 4, 6, len(capas) - 2]))
    captura = {}
    ganchos = [capas[i].register_forward_hook(lambda mod, ent, sal, i=i: captura.__setitem__(i, sal.detach().float().cpu().numpy()[0]))
               for i in indices]
    p0 = next(red.parameters())
    x = torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1)[None]).astype(np.float32) / 255).to(p0.device, p0.dtype)
    with torch.no_grad():
        sal = red(x)
    for h in ganchos:
        h.remove()
    sal = sal[0] if isinstance(sal, (list, tuple)) else sal
    prob = sal.detach().float().cpu().numpy().ravel()
    nombres = {int(k): v for k, v in m.names.items()}
    info = [{'capa': int(i), 'tipo': type(capas[i]).__name__, 'canales': int(captura[i].shape[0]),
             'alto': int(captura[i].shape[1]), 'ancho': int(captura[i].shape[2])} for i in indices]
    VALORES['mapas'] = {'imagen': ruta.name, 'n_capas': len(capas), 'tipos': [type(c).__name__ for c in capas], 'capas': info,
                        'prediccion': nombres[int(prob.argmax())], 'probabilidades': {nombres[k]: round(float(v), 4) for k, v in enumerate(prob)},
                        'parametros': int(sum(q.numel() for q in red.parameters()))}
    print(json.dumps(VALORES['mapas'], ensure_ascii=False)[:900])
    n_col = 6
    fig = plt.figure(figsize=(ANCHO_IN, 1.02 * (len(indices) + 1.15)))
    gs = fig.add_gridspec(len(indices) + 1, n_col + 1, width_ratios=[1.25] + [1] * n_col, wspace=0.06, hspace=0.12,
                          height_ratios=[1.15] + [1] * len(indices))
    ax = fig.add_subplot(gs[0, 1]); ax.imshow(rgb); sin_ejes(ax)
    ax = fig.add_subplot(gs[0, 0]); ax.axis('off')
    ax.text(1.0, 0.5, 'Entrada\n3 canales\n384 × 384', ha='right', va='center', fontsize=7, transform=ax.transAxes)
    for r, d in enumerate(info, 1):
        a = captura[d['capa']]
        orden = np.argsort(-a.reshape(a.shape[0], -1).std(1))[:n_col]
        ax = fig.add_subplot(gs[r, 0]); ax.axis('off')
        ax.text(1.0, 0.5, f"Capa {d['capa']}\n{d['canales']} canales\n{d['ancho']} × {d['alto']}", ha='right', va='center',
                fontsize=7, transform=ax.transAxes)
        for c, k in enumerate(orden):
            ax = fig.add_subplot(gs[r, c + 1])
            f = a[k]; f = (f - f.min()) / (f.max() - f.min() + 1e-9)
            ax.imshow(f, cmap='gray', interpolation='nearest'); sin_ejes(ax)
    guardar(fig, DIR_FIG / 'Fig_6_3_mapas_activacion.png'); plt.show()

seccion('mapas', s_mapas)

## 3. La salida del clasificador y la del detector

In [ ]:
def s_clasificador():
    m = YOLO(str(MOD_PALMA))
    rutas = sorted(p for p in (DATOS / 'test').glob('*/*') if p.suffix.lower() in EXT_IMG)
    res = m.predict(source=[str(p) for p in rutas], imgsz=384, batch=8, device=DISP, verbose=False)
    nombres = {int(k): v for k, v in res[0].names.items()}
    P = np.array([r.probs.data.detach().cpu().numpy().astype(float) for r in res])
    real = np.array([p.parent.name for p in rutas]); pred = np.array([nombres[int(k)] for k in P.argmax(1)]); conf = P.max(1)
    bien = real == pred
    elegidas = [int(np.where(bien)[0][np.argmax(conf[bien])]), int(np.where(bien)[0][np.argmin(conf[bien])])]
    if (~bien).any():
        elegidas.append(int(np.where(~bien)[0][np.argmax(conf[~bien])]))
    rotulos = ['Acierto con confianza alta', 'Acierto con confianza baja', 'Error con confianza alta']
    fig = plt.figure(figsize=(ANCHO_IN, 3.5))
    gs = fig.add_gridspec(2, len(elegidas), wspace=0.42, hspace=0.3, height_ratios=[1.25, 1])
    det = []
    for k, e in enumerate(elegidas):
        ax = fig.add_subplot(gs[0, k]); ax.imshow(recorte_cuadrado(leer_rgb(rutas[e], 640), 320)); sin_ejes(ax)
        ax.set_title(rotulos[k], loc='left', fontsize=7, fontweight='bold')
        ax.set_xlabel(f'Clase real: {NOMBRE_ES[real[e]].lower()}', fontsize=6.5, color=TINTA_2)
        ax = fig.add_subplot(gs[1, k])
        y = np.arange(len(CLASES))[::-1]
        v = [P[e][[i for i, n_ in nombres.items() if n_ == c][0]] for c in CLASES]
        ax.barh(y, v, 0.62, color=[ACENTO if c == pred[e] else GRIS_FONDO for c in CLASES])
        for yy, vv in zip(y, v):
            ax.text(min(vv + 0.03, 0.8), yy, coma(vv, 2), va='center', fontsize=6)
        ax.set_yticks(y); ax.set_yticklabels([NOMBRE_ES[c] for c in CLASES], fontsize=6); ax.set_xlim(0, 1); ax.set_xticks([0, 0.5, 1])
        ax.tick_params(axis='y', length=0); eje_con_coma(ax.xaxis, 1); ax.set_xlabel('Probabilidad', fontsize=6.5)
        det.append({'rotulo': rotulos[k], 'imagen': rutas[e].name, 'real': real[e], 'pred': pred[e],
                    'probabilidades': {nombres[i]: round(float(P[e][i]), 4) for i in nombres}})
    guardar(fig, DIR_FIG / 'Fig_6_5_salida_clasificador.png'); plt.show()
    VALORES['clasificador'] = {'ejemplos': det, 'exactitud_prueba': float(bien.mean()), 'errores': int((~bien).sum()),
                               'confianza_media_aciertos': float(conf[bien].mean()),
                               'confianza_media_errores': float(conf[~bien].mean()) if (~bien).any() else None,
                               'f1_macro': float(f1_score(real, pred, labels=CLASES_ORD, average='macro', zero_division=0))}
    pd.DataFrame({'imagen': [p.name for p in rutas], 'real': real, 'pred': pred, 'confianza': conf}).to_csv(DIR_VAL / 'cap06_predicciones_prueba_palma.csv', index=False)
    print(json.dumps(VALORES['clasificador'], ensure_ascii=False)[:700])

seccion('clasificador', s_clasificador)

In [ ]:
def iou_matriz(a, b):
    # IoU entre cada caja de a (n × 4) y cada caja de b (m × 4), en formato x1, y1, x2, y2.
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return (inter / (area_a[:, None] + area_b[None, :] - inter + 1e-12)).astype(np.float32)


def leer_cajas(ruta_txt, ancho=1.0, alto=1.0):
    # Lee un archivo YOLO y devuelve las cajas como x1, y1, x2, y2; descarta las de ancho o alto nulo.
    filas = [l.split() for l in Path(ruta_txt).read_text(encoding='utf-8', errors='ignore').splitlines() if len(l.split()) == 5]
    if not filas:
        return np.zeros((0, 4), np.float32)
    a = np.array(filas, dtype=np.float32)[:, 1:]
    c = np.column_stack([(a[:, 0] - a[:, 2] / 2) * ancho, (a[:, 1] - a[:, 3] / 2) * alto, (a[:, 0] + a[:, 2] / 2) * ancho, (a[:, 1] + a[:, 3] / 2) * alto])
    return c[(c[:, 2] > c[:, 0]) & (c[:, 3] > c[:, 1])]


def aciertos(M, conf, umbral):
    # Emparejamiento voraz: de mayor a menor confianza, cada detección toma la anotación libre con mayor IoU.
    tp = np.zeros(len(conf), bool); libre = np.ones(M.shape[1], bool)
    if M.shape[1] == 0:
        return tp
    for k in np.argsort(-conf):
        fila = np.where(libre, M[k], -1.0); j = int(fila.argmax())
        if fila[j] >= umbral:
            tp[k] = True; libre[j] = False
    return tp


def precision_media(conf, tp, n_gt):
    # Precisión media con interpolación de 101 puntos, como en COCO.
    orden = np.argsort(-conf); tp = tp[orden]
    acum_tp = np.cumsum(tp); acum_fp = np.cumsum(~tp)
    sens = acum_tp / max(n_gt, 1); prec = acum_tp / np.maximum(acum_tp + acum_fp, 1)
    env = np.maximum.accumulate(prec[::-1])[::-1]
    r = np.linspace(0, 1, 101); idx = np.searchsorted(sens, r, side='left')
    return float(np.where(idx < len(env), env[np.minimum(idx, len(env) - 1)], 0).mean()), sens, prec, conf[orden]


UMBRALES_IOU = np.round(np.arange(0.5, 0.96, 0.05), 2)


def s_detector():
    det = YOLO(str(MOD_PIO))
    rutas = sorted(p for p in (PIO / 'images' / 'val').iterdir() if p.suffix.lower() in EXT_IMG)
    gemelas = set()
    ruta_vec = DIR_VAL / 'pio_vecino_val_train.csv'
    if ruta_vec.exists():
        v = pd.read_csv(ruta_vec); gemelas = set(v.loc[v['dist'] < 1, 'archivo'])
    reg, guardadas = [], {}
    for k in range(0, len(rutas), 16):
        lote = rutas[k:k + 16]
        res = det.predict(source=[str(p) for p in lote], imgsz=960, conf=0.001, iou=0.7, max_det=1000, device=DISP, verbose=False)
        for ruta, r in zip(lote, res):
            alto, ancho = r.orig_shape
            cajas = r.boxes.xyxy.detach().cpu().numpy().astype(np.float32); conf = r.boxes.conf.detach().cpu().numpy().astype(np.float32)
            gt = leer_cajas(PIO / 'labels' / 'val' / (ruta.stem + '.txt'), ancho, alto)
            M = iou_matriz(cajas, gt)
            reg.append({'archivo': ruta.name, 'gemela': ruta.name in gemelas, 'n_gt': len(gt), 'conf': conf,
                        'tp': {float(u): aciertos(M, conf, u) for u in UMBRALES_IOU}})
            guardadas[ruta.name] = (ruta, cajas, conf, gt)
        print(f'Inferencia: {min(k + 16, len(rutas))}/{len(rutas)}', end='\r')
    print()

    def resumen(filtro):
        sel = [x for x in reg if filtro(x)]
        conf = np.concatenate([x['conf'] for x in sel]); n_gt = int(sum(x['n_gt'] for x in sel))
        ap, curvas = {}, {}
        for u in UMBRALES_IOU:
            tp = np.concatenate([x['tp'][float(u)] for x in sel])
            ap[float(u)], sens, prec, c_ord = precision_media(conf, tp, n_gt)
            if float(u) in (0.5, 0.75):
                paso = max(1, len(sens) // 400)
                curvas[float(u)] = {'sensibilidad': sens[::paso].round(4).tolist(), 'precision': prec[::paso].round(4).tolist(),
                                    'confianza': c_ord[::paso].round(4).tolist()}
        m25 = conf >= 0.25; tp50 = np.concatenate([x['tp'][0.5] for x in sel])
        return {'imagenes': len(sel), 'anotaciones': n_gt, 'detecciones': int(len(conf)), 'AP_por_umbral': ap,
                'AP50': ap[0.5], 'AP75': ap[0.75], 'AP50_95': float(np.mean(list(ap.values()))),
                'conf_0_25': {'detecciones': int(m25.sum()), 'precision': float(tp50[m25].mean()), 'sensibilidad': float(tp50[m25].sum() / n_gt)},
                'curvas': curvas}
    R = {'todas': resumen(lambda x: True)}
    if gemelas:
        R['sin_gemela'] = resumen(lambda x: not x['gemela']); R['gemelas'] = resumen(lambda x: x['gemela'])
    VALORES['detector'] = R
    print({k: {q: (round(v, 4) if isinstance(v, float) else v) for q, v in r.items() if q in ('imagenes', 'anotaciones', 'AP50', 'AP75', 'AP50_95')} for k, r in R.items()})

    # Figura: curva de precisión y sensibilidad con dos umbrales de IoU.
    fig, ax = plt.subplots(figsize=(ANCHO_IN * 0.62, 2.7))
    for u, color in ((0.5, ACENTO), (0.75, TINTA_2)):
        c = R['todas']['curvas'][u]
        ax.plot(c['sensibilidad'], c['precision'], color=color, lw=1.3, label=f"IoU ≥ {coma(u, 2)} · AP = {coma(R['todas']['AP_por_umbral'][u], 3)}")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_xlabel('Sensibilidad'); ax.set_ylabel('Precisión')
    ax.grid(True, color=REJILLA, lw=0.6); ax.set_axisbelow(True); eje_con_coma(ax.xaxis, 1); eje_con_coma(ax.yaxis, 1)
    ax.legend(frameon=False, loc='lower left')
    guardar(fig, DIR_FIG / 'Fig_6_9_precision_sensibilidad.png'); plt.show()

    # Figura: una escena con sus detecciones y un detalle con anotaciones, detecciones e IoU.
    cand = sorted((x for x in reg if not x['gemela']), key=lambda x: (abs(x['n_gt'] - np.median([y['n_gt'] for y in reg])), x['archivo']))
    ruta, cajas, conf, gt = guardadas[cand[0]['archivo']]
    img = leer_rgb(ruta); alto, ancho = img.shape[:2]
    sel = conf >= 0.25; cajas, conf = cajas[sel], conf[sel]
    M = iou_matriz(cajas, gt)
    fig = plt.figure(figsize=(ANCHO_IN, 2.75))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.78, 1], wspace=0.04, left=0.005, right=0.995, top=0.92, bottom=0.02)
    a1 = fig.add_subplot(gs[0, 0]); a1.imshow(img); sin_ejes(a1)
    for c in cajas:
        a1.add_patch(Rectangle((c[0], c[1]), c[2] - c[0], c[3] - c[1], fill=False, ec=ACENTO, lw=0.6))
    a1.set_title(f'a) {ruta.name}: {len(cajas)} detecciones, {len(gt)} aves anotadas', loc='left', fontweight='bold')
    lado = 0.26 * alto
    centros = (gt[:, :2] + gt[:, 2:]) / 2
    cx, cy = centros[np.argmin(np.hypot(centros[:, 0] - ancho / 2, centros[:, 1] - alto / 2))] if len(gt) else (ancho / 2, alto / 2)
    x0, y0 = int(np.clip(cx - lado / 2, 0, ancho - lado)), int(np.clip(cy - lado / 2, 0, alto - lado))
    a1.add_patch(Rectangle((x0, y0), lado, lado, fill=False, ec='white', lw=1.0, ls=(0, (3, 2))))
    a2 = fig.add_subplot(gs[0, 1]); a2.imshow(img[y0:y0 + int(lado), x0:x0 + int(lado)]); sin_ejes(a2)
    dentro = lambda c: (c[0] + c[2]) / 2 > x0 and (c[0] + c[2]) / 2 < x0 + lado and (c[1] + c[3]) / 2 > y0 and (c[1] + c[3]) / 2 < y0 + lado
    for g_ in gt:
        if dentro(g_):
            a2.add_patch(Rectangle((g_[0] - x0, g_[1] - y0), g_[2] - g_[0], g_[3] - g_[1], fill=False, ec='white', lw=1.1, ls=(0, (3, 2))))
    ious = []
    for k, c in enumerate(cajas):
        if dentro(c):
            a2.add_patch(Rectangle((c[0] - x0, c[1] - y0), c[2] - c[0], c[3] - c[1], fill=False, ec=ACENTO, lw=1.2))
            v = float(M[k].max()) if M.shape[1] else 0.0; ious.append(v)
            a2.text(c[0] - x0 + 2, c[1] - y0 + 2, coma(v, 2), fontsize=5.6, color='white', va='top',
                    bbox=dict(boxstyle='round,pad=0.12', fc=ACENTO, ec='none'))
    a2.set_title('b) Anotación, detección e IoU', loc='left', fontweight='bold')
    guardar(fig, DIR_FIG / 'Fig_6_6_salida_detector.png'); plt.show()
    VALORES['fig_detector'] = {'archivo': ruta.name, 'detecciones_conf_025': int(len(cajas)), 'anotadas': int(len(gt)),
                               'iou_en_el_detalle': [round(v, 3) for v in ious], 'confianza_mediana': float(np.median(conf)) if len(conf) else None}

seccion('detector', s_detector)

## 4. Acuerdo entre dos anotaciones de la misma imagen

In [ ]:
def s_acuerdo():
    from scipy.optimize import linear_sum_assignment
    v = pd.read_csv(DIR_VAL / 'pio_vecino_val_train.csv'); pares = v[v['dist'] < 1].reset_index(drop=True)
    ious, filas = [], []
    for _, f in pares.iterrows():
        a = leer_cajas(PIO / 'labels' / 'val' / (Path(f['archivo']).stem + '.txt'))
        b = leer_cajas(PIO / 'labels' / 'train' / (Path(f['vecino_train']).stem + '.txt'))
        M = iou_matriz(a, b)
        i, j = linear_sum_assignment(-M)
        ok = M[i, j] >= 0.1
        ious.extend(M[i, j][ok].tolist())
        filas.append({'val': f['archivo'], 'train': f['vecino_train'], 'cajas_val': len(a), 'cajas_train': len(b),
                      'emparejadas': int(ok.sum()), 'iou_mediana': float(np.median(M[i, j][ok])) if ok.any() else None})
    ious = np.array(ious); t = pd.DataFrame(filas); t.to_csv(DIR_VAL / 'cap06_acuerdo_anotaciones.csv', index=False)
    total = int(t['cajas_val'].sum() + t['cajas_train'].sum())
    VALORES['acuerdo'] = {'pares_de_imagenes': int(len(t)), 'cajas_val': int(t['cajas_val'].sum()), 'cajas_train': int(t['cajas_train'].sum()),
                          'emparejadas': int(len(ious)), 'sin_pareja': total - 2 * int(len(ious)),
                          'iou': {'media': float(ious.mean()), 'p5': float(np.percentile(ious, 5)), 'p25': float(np.percentile(ious, 25)),
                                  'mediana': float(np.median(ious)), 'p75': float(np.percentile(ious, 75))},
                          'fraccion_iou_ge': {str(u): float((ious >= u).mean()) for u in (0.5, 0.75, 0.9, 0.95, 0.999)},
                          'histograma': np.histogram(ious, bins=np.linspace(0, 1, 41))[0].tolist()}
    print(json.dumps({k: v_ for k, v_ in VALORES['acuerdo'].items() if k != 'histograma'}, indent=1))
    # Figura: las dos anotaciones sobre un detalle de la imagen con más discrepancias entre copias.
    f = t.assign(d=(t['cajas_val'] - t['cajas_train']).abs()).sort_values(['d', 'val'], ascending=[False, True]).iloc[0]
    ruta = PIO / 'images' / 'val' / f['val']; img = leer_rgb(ruta); alto, ancho = img.shape[:2]
    a = leer_cajas(PIO / 'labels' / 'val' / (Path(f['val']).stem + '.txt'), ancho, alto)
    b = leer_cajas(PIO / 'labels' / 'train' / (Path(f['train']).stem + '.txt'), ancho, alto)
    fig = plt.figure(figsize=(ANCHO_IN, 2.7))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.45, 1], wspace=0.3)
    ax = fig.add_subplot(gs[0, 0])
    x0, x1, y0, y1 = int(ancho * 0.25), int(ancho * 0.75), int(alto * 0.2), int(alto * 0.8)
    ax.imshow(img[y0:y1, x0:x1]); sin_ejes(ax)
    for c, estilo in ((a, dict(ec=ACENTO, lw=1.0)), (b, dict(ec='white', lw=0.9, ls=(0, (3, 2))))):
        for q in c:
            if x0 < (q[0] + q[2]) / 2 < x1 and y0 < (q[1] + q[3]) / 2 < y1:
                ax.add_patch(Rectangle((q[0] - x0, q[1] - y0), q[2] - q[0], q[3] - q[1], fill=False, **estilo))
    ax.set_title(f"a) Dos anotaciones de un mismo fotograma", loc='left', fontweight='bold')
    ax.set_xlabel(f"Línea continua: {Path(f['val']).stem}, {int(f['cajas_val'])} cajas\nLínea discontinua: {Path(f['train']).stem}, {int(f['cajas_train'])} cajas", fontsize=6, color=TINTA_2)
    ax = fig.add_subplot(gs[0, 1])
    ax.hist(ious, bins=np.linspace(0, 1, 41), color=ACENTO)
    ax.set_xlabel('IoU entre las dos anotaciones'); ax.set_ylabel('Número de aves')
    ax.yaxis.grid(True, color=REJILLA, lw=0.6); ax.set_axisbelow(True); ax.set_xticks([0, 0.25, 0.5, 0.75, 1]); eje_con_coma(ax.xaxis, 2)
    ax.set_title('b) Acuerdo entre anotaciones', loc='left', fontweight='bold')
    guardar(fig, DIR_FIG / 'Fig_6_8_acuerdo_anotaciones.png'); plt.show()
    VALORES['fig_acuerdo'] = {'val': f['val'], 'train': f['train'], 'cajas_val': int(f['cajas_val']), 'cajas_train': int(f['cajas_train'])}

seccion('acuerdo', s_acuerdo)

## 5. Transferencia de aprendizaje: preentrenado frente a entrenado desde cero

In [ ]:
LOCAL_DATOS = LOCAL / 'PALMA_CAP06'


def evaluar(modelo, particion):
    rutas = sorted(p for p in (LOCAL_DATOS / particion).glob('*/*') if p.suffix.lower() in EXT_IMG)
    res = modelo.predict(source=[str(p) for p in rutas], imgsz=384, batch=8, device=DISP, verbose=False)
    nombres = {int(k): v for k, v in res[0].names.items()}
    real = [p.parent.name for p in rutas]; pred = [nombres[int(np.argmax(r.probs.data.detach().cpu().numpy()))] for r in res]
    return {'exactitud': float(accuracy_score(real, pred)),
            'f1_macro': float(f1_score(real, pred, labels=CLASES_ORD, average='macro', zero_division=0))}


def corrida(nombre, pesos, semilla, epocas, paciencia, preentrenado):
    ruta_json, ruta_csv = DIR_EXP / f'{nombre}.json', DIR_EXP / f'{nombre}_curva.csv'
    if ruta_json.exists() and ruta_csv.exists():
        print('REUTILIZADA:', nombre); return json.loads(ruta_json.read_text(encoding='utf-8'))
    random.seed(semilla); np.random.seed(semilla); torch.manual_seed(semilla); torch.cuda.manual_seed_all(semilla)
    t0 = time.time()
    YOLO(pesos).train(data=str(LOCAL_DATOS), epochs=epocas, imgsz=384, batch=8, patience=paciencia, device=DISP, workers=2,
                      seed=semilla, deterministic=True, pretrained=preentrenado, optimizer='auto', amp=True, cache=True,
                      project=str(LOCAL / 'cap06_runs'), name=nombre, exist_ok=True, plots=False, verbose=False)
    minutos = (time.time() - t0) / 60
    carpeta = LOCAL / 'cap06_runs' / nombre
    curva = pd.read_csv(carpeta / 'results.csv'); curva.columns = [c.strip() for c in curva.columns]
    modelo = YOLO(str(carpeta / 'weights' / 'best.pt'))
    r = {'nombre': nombre, 'pesos': pesos, 'semilla': semilla, 'epocas_max': epocas, 'paciencia': paciencia, 'preentrenado': preentrenado,
         'epocas_hechas': int(len(curva)), 'minutos': round(minutos, 2), 'val': evaluar(modelo, 'val'), 'prueba': evaluar(modelo, 'test'),
         'columnas_curva': list(curva.columns)}
    curva.to_csv(ruta_csv, index=False); ruta_json.write_text(json.dumps(r, ensure_ascii=False, indent=1), encoding='utf-8')
    print(f"FINALIZADA: {nombre} | {minutos:.1f} min | {len(curva)} épocas | val F1 {r['val']['f1_macro']:.3f} | prueba F1 {r['prueba']['f1_macro']:.3f}")
    del modelo; torch.cuda.empty_cache() if HAY_GPU else None; gc.collect()
    return r


def s_transferencia():
    if LOCAL_DATOS.exists():
        shutil.rmtree(LOCAL_DATOS)
    shutil.copytree(DATOS, LOCAL_DATOS)
    plan = [('pre_seed2026', 'yolo11m-cls.pt', 2026, 60, 15, True)]
    plan += [(f'cero60_seed{s}', 'yolo11m-cls.yaml', s, 60, 15, False) for s in (2026, 2027, 2028)]
    plan += [(f'cero200_seed{s}', 'yolo11m-cls.yaml', s, 200, 50, False) for s in (2026, 2027, 2028)]
    R = []
    for p in plan:
        R.append(corrida(*p))
        VALORES['transferencia'] = {'corridas': R, 'f1_preentrenado_proyecto': {2026: 0.8667, 2027: 0.8103, 2028: 0.8615}}
    t = pd.DataFrame([{'nombre': r['nombre'], 'épocas': r['epocas_hechas'], 'min': r['minutos'], 'val_f1': r['val']['f1_macro'],
                       'prueba_f1': r['prueba']['f1_macro'], 'prueba_exactitud': r['prueba']['exactitud']} for r in R])
    print(t.round(3).to_string(index=False))
    VALORES['transferencia']['medias'] = {g: {'prueba_f1': float(t[t['nombre'].str.startswith(g)]['prueba_f1'].mean()),
                                              'val_f1': float(t[t['nombre'].str.startswith(g)]['val_f1'].mean()),
                                              'epocas': float(t[t['nombre'].str.startswith(g)]['épocas'].mean())} for g in ('pre', 'cero60', 'cero200')}
    # Figura: curvas de validación de la semilla 2026.
    fig, ejes = plt.subplots(1, 2, figsize=(ANCHO_IN, 2.5), gridspec_kw={'wspace': 0.28})
    for nombre, rot, color, ls in (('pre_seed2026', 'Preentrenado', ACENTO, '-'), ('cero200_seed2026', 'Desde cero', TINTA_2, '-')):
        c = pd.read_csv(DIR_EXP / f'{nombre}_curva.csv')
        col_acc = [k for k in c.columns if 'accuracy_top1' in k][0]; col_loss = [k for k in c.columns if k.startswith('train/loss')][0]
        ejes[0].plot(c['epoch'], c[col_loss], color=color, lw=1.2, ls=ls, label=rot)
        ejes[1].plot(c['epoch'], c[col_acc], color=color, lw=1.2, ls=ls, label=rot)
    ejes[0].set_ylabel('Pérdida en entrenamiento'); ejes[1].set_ylabel('Exactitud en validación'); ejes[1].set_ylim(0, 1)
    for ax, tit in zip(ejes, ('a) Pérdida', 'b) Exactitud en validación')):
        ax.set_xlabel('Época'); ax.grid(True, color=REJILLA, lw=0.6); ax.set_axisbelow(True); ax.set_title(tit, loc='left', fontweight='bold')
        eje_con_coma(ax.yaxis, 1)
    ejes[1].legend(frameon=False, loc='lower right')
    guardar(fig, DIR_FIG / 'Fig_6_4_transferencia.png'); plt.show()

seccion('transferencia', s_transferencia)

## 6. Tamaño de los modelos YOLO11 y YOLO26

In [ ]:
def s_familia():
    filas = []
    for pesos in ('yolo11n-cls.pt', 'yolo11s-cls.pt', 'yolo11m-cls.pt', 'yolo26n-cls.pt', 'yolo26s-cls.pt', 'yolo26m-cls.pt',
                  'yolo11n.pt', 'yolo11s.pt', 'yolo11m.pt', 'yolo26n.pt', 'yolo26s.pt', 'yolo26m.pt'):
        try:
            m = YOLO(pesos); red = m.model
            tipos = Counter(type(x).__name__ for x in red.modules())
            fila = {'pesos': pesos, 'tarea': m.task, 'parametros': int(sum(p.numel() for p in red.parameters())),
                    'capas_principales': [type(x).__name__ for x in red.model], 'n_modulos': int(sum(tipos.values())),
                    'modulos_frecuentes': dict(tipos.most_common(8)), 'archivo_mb': round(Path(pesos).stat().st_size / 1e6, 2) if Path(pesos).exists() else None}
            try:
                inf = m.info(verbose=False)
                fila['info'] = [float(v) for v in inf] if inf is not None else None
            except Exception as e:
                fila['info'] = 'error: ' + str(e)[:80]
            filas.append(fila)
            print(pesos, fila['parametros'], fila['info'], fila['capas_principales'])
        except Exception as e:
            filas.append({'pesos': pesos, 'error': str(e)[:200]}); print(pesos, 'ERROR', str(e)[:120])
    VALORES['familia'] = filas

seccion('familia', s_familia)

## 7. Guardar y descargar

In [ ]:
VALORES['errores'] = ERRORES
VALORES['entorno'] = {'gpu': GPU, 'torch': torch.__version__, 'ultralytics': ultralytics.__version__}
ruta_json = DIR_VAL / 'valores_cap06.json'
ruta_json.write_text(json.dumps(VALORES, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
ruta_zip = LOCAL / 'AgroVision_cap06.zip'
with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DIR_FIG.glob('Fig_6_*.png')):
        z.write(f, 'cap06/' + f.name)
    for f in [ruta_json, DIR_VAL / 'cap06_predicciones_prueba_palma.csv', DIR_VAL / 'cap06_acuerdo_anotaciones.csv'] + sorted(DIR_EXP.glob('*')):
        if f.exists():
            z.write(f, f.name)
print('Secciones con error:', list(ERRORES) or 'ninguna')
print('Figuras:', len(FIGURAS), '· zip de', round(ruta_zip.stat().st_size / 1e6, 1), 'MB')
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))